In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import ants
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
import marimo as mo
os.chdir(mo.notebook_dir()) #Jupyterlab-like, change path to where the notebook is, all paths relative to this

In [ ]:
import deepcor

In [ ]:
deepcor.utils.check_gpu_and_speedup()

In [ ]:
# ModelConfig: Configure the model architecture
model_config = deepcor.config.ModelConfig(
    latent_dims=(8, 8),  # (shared dim, specific dim)
    beta=0.01,           # KLD loss weight
    gamma=0.0,           # TC loss weight
    delta=0.0,           # RONI zero constraint weight
    scale_MSE_GM=1e3,    # Gray matter reconstruction loss scale
    scale_MSE_CF=1e3,    # Non-gray matter reconstruction loss scale
    scale_MSE_FG=0.0,    # Foreground reconstruction loss scale
    do_disentangle=True  # Enable disentanglement
)


# TrainingConfig: Configure training parameters
training_config = deepcor.config.TrainingConfig(
    n_epochs=100,
    batch_size=1024,
    learning_rate=0.001,
    optimizer='adamw',
    betas=(0.9, 0.999),
    eps=1e-08,
    max_grad_norm=5.0,
    n_repetitions=20  # Number of ensemble repetitions
)


# DataConfig: Configure data preprocessing
data_config = deepcor.config.DataConfig(
    n_dummy_scans=0,
    apply_censoring=False,
    censoring_threshold=0.5,
    confound_columns=['X', 'Y', 'Z', 'RotX', 'RotY', 'RotZ']
)

# Create a complete configuration
config = deepcor.config.DeepCorConfig(
    model=model_config,
    training=training_config,
    data=data_config
)


print(model_config);print('\n')
print(training_config);print('\n')
print(data_config);print('\n')
print(config);print('\n')

ModelConfig(latent_dims=(8, 8), hidden_dims=None, beta=0.01, gamma=0.0, delta=0.0, scale_MSE_GM=1000.0, scale_MSE_CF=1000.0, scale_MSE_FG=0.0, do_disentangle=True)


TrainingConfig(n_epochs=100, batch_size=1024, learning_rate=0.001, optimizer='adamw', betas=(0.9, 0.999), eps=1e-08, max_grad_norm=5.0, n_repetitions=20)


DataConfig(n_dummy_scans=0, apply_censoring=False, censoring_threshold=0.5, also_nearby_voxels=True, confound_columns=['X', 'Y', 'Z', 'RotX', 'RotY', 'RotZ'])


DeepCorConfig(model=ModelConfig(latent_dims=(8, 8), hidden_dims=None, beta=0.01, gamma=0.0, delta=0.0, scale_MSE_GM=1000.0, scale_MSE_CF=1000.0, scale_MSE_FG=0.0, do_disentangle=True), training=TrainingConfig(n_epochs=100, batch_size=1024, learning_rate=0.001, optimizer='adamw', betas=(0.9, 0.999), eps=1e-08, max_grad_norm=5.0, n_repetitions=20), data=DataConfig(n_dummy_scans=0, apply_censoring=False, censoring_threshold=0.5, also_nearby_voxels=True, confound_columns=['X', 'Y', 'Z', 'RotX', 'RotY', 'RotZ']))




In [ ]:
# Define Data Paths
# Cell Tagged parameters for papermill looping


bids_path = '../Data/fMRI-Data/studyforrest-fmriprep/'

subs = [sub for sub in os.listdir(os.path.join(bids_path)) if sub.startswith('sub-')]
subs.sort()

session = 'ses-localizer'
task = 'objectcategories'
space = 'MNI152NLin2009cAsym'

s = 0
r = 1
analysis_name = 'test-advanced'